# 06 — Agent Memory

**Mode:** Offline  
**Level:** Fundamentals  
**Estimated time:** 30 minutes

[Watch the original lesson video](https://www.youtube.com/watch?v=sH3NkfoTZ1c)

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## What you will build

A memory-enabled Agent with short-term, episodic, and semantic entries. You will inspect the manager, recall relevant content, resolve a memory by ID, and see where persistent long-term storage enters the architecture.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, get_events, require_env, setup_notebook,
    show_callout, show_flow, show_json, show_roles, show_spore,
    show_timeline, stage,
)

setup_notebook(6, 'Agent Memory')


## Prerequisites and setup

**Course prerequisites:** Complete `course-01-hello-world`.

**Execution requirements:** Offline. The in-memory backend requires no embedding model. Lesson 07 moves the same public API to persistent Qdrant vector memory.

Run the setup cell above first. It configures presentation helpers and a
credential-free lifecycle provider. It does not hide any agent workflow.


## Learning goals

- Distinguish short-term, episodic, semantic, and long-term memory roles.
- Store typed memories with importance.
- Recall by query and by stable reference ID.
- Inspect and clear agent-scoped memory state.


## Mental model

**Short-term memory** keeps recent working context. **Episodic memory** records what happened. **Semantic memory** records reusable facts and concepts. **Long-term memory** describes persistence beyond the current process and often uses embeddings plus a vector store. Memory type answers what the knowledge means; backend answers where it is stored.


In [ ]:
show_flow(
('Agent', 'remember', 'agent'),
('Memory manager', 'type and scope', 'memory'),
('Backend', 'store entries', 'memory'),
('Recall', 'rank relevant', 'agent'),
)


## Try it

We will now assemble the workflow in small steps. Run each cell, then pause at the inspection that follows it.


### Create a memory-enabled Agent

The explicit `memory` backend gives this lesson real memory behavior without a remote vector database or embedding API.


In [ ]:
from praval import Agent

learner = Agent(
    "course-memory-agent",
    provider="notebook-lifecycle",
    model="notebook-lifecycle-model",
    memory_enabled=True,
    memory_config={"backend": "memory"},
)
assert learner.memory is not None
stage("agent", "memory initialized", learner.memory.backend)


### What just happened?

The Agent owns a MemoryManager scoped to `course-memory-agent`. Its short-term store is immediately available.

### Why this matters

Agent scoping prevents one participant's working state from becoming another's by accident.


In [ ]:
show_json(learner.memory.get_active_backend(), "Active memory backend")


### Store three kinds of memory

Each entry has content, importance, and a semantic type. The returned ID can be carried in a lightweight Spore reference.


In [ ]:
memories = [
    ("The learner is exploring Praval memory.", 0.5, "short_term"),
    ("The learner completed the Reef architecture exercise.", 0.8, "episodic"),
    ("Spores carry structured knowledge between agents.", 0.95, "semantic"),
]
memory_ids = {}
for content, importance, memory_type in memories:
    memory_id = learner.remember(
        content, importance=importance, memory_type=memory_type
    )
    assert memory_id
    memory_ids[memory_type] = memory_id
    stage("memory", "stored", f"{memory_type}: {content[:32]}")


### What just happened?

The backend stored three MemoryEntry objects. Their stable IDs are separate from their natural-language content.

### Why this matters

Type and importance give later retrieval and retention policies useful structure.


In [ ]:
show_json(
    {
        "ids": memory_ids,
        "stats": learner.memory.get_memory_stats(),
    },
    "Memory manager state",
)


### Recall by meaning

The in-memory backend uses its local search path. The same `Agent.recall()` call will use vector similarity when lesson 07 switches backends.


In [ ]:
recalled = learner.recall("Spores carry structured knowledge", limit=3)
stage("memory", "recalled", f"{len(recalled)} entries")

assert recalled
assert any("Spores" in entry.content for entry in recalled)
show_json(
    [
        {
            "id": entry.id,
            "type": entry.memory_type.value,
            "content": entry.content,
            "importance": entry.importance,
        }
        for entry in recalled
    ],
    "Ranked recall",
)


### What just happened?

Recall returned MemoryEntry objects rather than plain strings, so identity, type, importance, and content remain available.

### Why this matters

Applications can use memory metadata to decide what may enter a prompt or a Spore.


### Resolve a stable memory reference

A reference ID lets a Spore point to stored knowledge without repeating a large payload. Resolution remains scoped to the memory manager.


In [ ]:
semantic_id = memory_ids["semantic"]
resolved = learner.recall_by_id(semantic_id)
assert len(resolved) == 1
assert resolved[0].content.startswith("Spores carry")

show_json(
    {"reference": semantic_id, "resolved_content": resolved[0].content},
    "Memory reference resolution",
)
show_timeline()


### What just happened?

The ID resolved to the original typed entry without a free-text search.

### Why this matters

References are useful when message size, privacy boundaries, or shared knowledge stores make full content duplication undesirable.


### Place long-term memory in the picture

The current backend is process-local. Persistent semantic and episodic entries need an embedding runtime and durable vector store.


In [ ]:
backend = learner.memory.get_active_backend()
persistence_plan = {
    "current": backend["name"],
    "persistent": backend["type"] == "persistent",
    "next_backend": "qdrant",
    "same_agent_methods": ["remember", "recall", "recall_by_id"],
}
assert persistence_plan["persistent"] is False
show_json(persistence_plan, "From local to long-term memory")


### What just happened?

The public Agent methods do not change when storage becomes persistent. The configuration and operational responsibilities do.

### Why this matters

A stable API keeps workflows focused while still making backend choice explicit.


## Your turn

Store one preference as semantic memory and one completed action as episodic memory. Recall both and compare their `memory_type` values.


In [ ]:
# preference_id = learner.remember(
#     "The learner prefers concise summaries.", memory_type="semantic"
# )
# action_id = learner.remember(
#     "The learner ran lesson 06.", memory_type="episodic"
# )
# Inspect learner.recall_by_id(preference_id) and recall_by_id(action_id).


## Common mistake

**Mistake:** Putting every conversation token into permanent semantic memory.

**Correction:** Classify memory deliberately. Keep transient context short-lived and promote only stable, useful knowledge to persistent stores.


<details>
<summary>Under the hood</summary>

MemoryManager always has short-term storage. Persistent backends add vector storage, and specialized episodic/semantic managers build policies above it. Search results are combined and deduplicated before the Agent returns them.

</details>


## Recap

- Memory type describes meaning; backend describes storage.
- Entries retain identity, content, importance, and type.
- Recall supports semantic queries and exact reference resolution.
- Persistent long-term memory adds embeddings and operational cleanup.


## Cleanup

Release every agent, transport, temporary file, or external resource created by this lesson. Cleanup is part of the example, not an afterthought.


In [ ]:
learner.memory.clear_agent_memories(learner.name)
learner.close()
stage("agent", "closed", "memory resources released")
assert learner._closed


### Next lesson

Continue to `07_qdrant_integration.ipynb` to store semantic memories with real provider embeddings in a real Qdrant collection.
